# GraphKV Gemini answer-quality experiment


Runs the repaired CPU/API evaluator for HotpotQA and 2WikiMultiHopQA. It compares cosine, fixed graph, offline global adaptation, and online warm adaptation under an equal reference-token budget. API keys are requested privately and are never written to the notebook.


In [ ]:
from pathlib import Path
import getpass, os, subprocess, sys

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
API = ROOT / 'api_quality'
assert (API / 'run_quality.py').exists(), API
os.chdir(API)
print('Working directory:', Path.cwd())


In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt'], check=True)


In [ ]:
if not os.environ.get('GEMINI_API_KEY'):
    os.environ['GEMINI_API_KEY'] = getpass.getpass('Gemini API key: ')
print('API key loaded:', bool(os.environ.get('GEMINI_API_KEY')))


Use the exact stable model ID available to the project. The completed paper runs used the reported identifier below. A different model ID requires a new output directory.


In [ ]:
MODEL = 'gemini-3.5-flash-lite'
subprocess.run([sys.executable, 'run_quality.py', '--list-models'], check=True)


## HotpotQA: 120 train / 40 development / 60 held-out questions


In [ ]:
HOT_OUTPUT = 'outputs/gemini_hotpot60_K10'
hotpot_cmd = [sys.executable, 'run_quality.py', '--dataset', 'hotpot',
    '--model', MODEL, '--n-train', '120', '--n-dev', '40', '--n-test', '60',
    '--k-values', '10', '--top-m', '5', '--context-token-budget', '768',
    '--max-output-tokens', '64', '--seed', '42', '--rpm', '0',
    '--output-dir', HOT_OUTPUT]
subprocess.run(hotpot_cmd, check=True)


## 2WikiMultiHopQA with the same settings


In [ ]:
WIKI_OUTPUT = 'outputs/gemini_2wiki60_K10'
wiki_cmd = [sys.executable, 'run_quality.py', '--dataset', '2wiki',
    '--model', MODEL, '--n-train', '120', '--n-dev', '40', '--n-test', '60',
    '--k-values', '10', '--top-m', '5', '--context-token-budget', '768',
    '--max-output-tokens', '64', '--seed', '42', '--rpm', '0',
    '--output-dir', WIKI_OUTPUT]
subprocess.run(wiki_cmd, check=True)


In [ ]:
import pandas as pd
for output in [HOT_OUTPUT, WIKI_OUTPUT]:
    print(output)
    display(pd.read_csv(Path(output) / 'summary.csv'))


If quota interrupts a run, execute the identical command again. The evaluator resumes completed policy calls and writes `COMPLETE.json` only when the run is complete.
